# Week-2: Provider-Agnostic LLM Client
One `LLMClient` interface over **Gemini** (hosted) and **Ollama + Llama 3.2** (local), running fully in Colab.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower). Add your key in the 🔑 **Secrets** sidebar as `GEMINI_API_KEY` and turn on *Notebook access*. Run cells in order.

## Step 1: Project folder

In [1]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


## Step 2: Install Ollama

In [2]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 3: Start the Ollama server in the background

In [3]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(["ollama", "serve"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


## Step 4: Download Llama 3.2 (~2 GB)

In [4]:
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


## Step 5: Quick test of the local model

In [5]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-16T08:03:08.286807435Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":108948461053,"load_duration":57502507502,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":51404105000,"eval_count":3,"eval_duration":38907000}

## Step 6: Save the Gemini key into `.env`
Get a key at aistudio.google.com/apikey and store it in Colab Secrets first.

In [6]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")
with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')
print(".env created. Key length:", len(key))

.env created. Key length: 53


In [7]:
!ls -a

.  ..  .env


## Step 7: requirements.txt and dependencies

In [8]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [9]:
!pip install -q -r requirements.txt
from google import genai
print('google-genai imported OK')

google-genai imported OK


## Step 8: `llm_client.py`
> The original notes used `gemini-3.8-flash`, which is not in Google's current model list. This uses **`gemini-3.7-flash`** (latest Flash at time of writing). If it errors, try `gemini-3.6-flash`.

In [10]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests

# Common interface: every provider must implement generate()
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass

# Gemini cloud provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.7-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text

# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={"model": self.model, "prompt": prompt, "stream": False},
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]

# Common client: the rest of your program only talks to this class
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Writing llm_client.py


## Step 9: `main.py`

In [11]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import GeminiProvider, OllamaProvider, LLMClient

load_dotenv(override=True)

provider_name = input("Choose provider (gemini/ollama): ").strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

Writing main.py


## Step 10: Unit test with a fake provider

In [12]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient

# Fake provider for testing
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."

class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        client = LLMClient(FakeProvider())
        result = client.generate("Explain AI")
        self.assertEqual(result, "AI stands for Artificial Intelligence.")

if __name__ == "__main__":
    unittest.main()

Writing test_client.py


In [13]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: anyio-4.16.0, platformdirs-4.14.3, langsmith-0.14.17, typeguard-4.6.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.49s ===============================


## Step 11: Run the program
Use `%run` so `input()` prompts work. Run it once with **ollama** and once with **gemini**, asking the same question (e.g. *Explain AI in two sentences*).

In [14]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: Explain AI in two sentences).

LLM Response:
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making, by using algorithms and data to make predictions or take actions. AI has the potential to automate various tasks, improve efficiency, and enhance productivity, but it also raises concerns about job displacement, data privacy, and bias in decision-making.


In [15]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: hello

LLM Response:
Hello! How can I assist you today?


### Optional: prove it's provider-agnostic in one cell

In [ ]:

import time
from llm_client import GeminiProvider, OllamaProvider, LLMClient
from google.colab import userdata

q = "Explain AI in two sentences."

# Ollama
print("\n--- OLLAMA ---")
try:
    print(LLMClient(OllamaProvider()).generate(q))
except Exception as e:
    print("Ollama Error:", e)

# Gemini
print("\n--- GEMINI ---")
try:
    gemini = GeminiProvider(userdata.get("GEMINI_API_KEY"))
    print(LLMClient(gemini).generate(q))

except Exception as e:
    print("Gemini is currently unavailable.")
    print("Error:", e)




--- OLLAMA ---
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. These systems use complex algorithms and data to simulate human-like cognitive abilities, enabling them to analyze vast amounts of data, recognize patterns, and make predictions or take actions autonomously.

--- GEMINI ---
